In [8]:
from transformers import AutoTokenizer, AutoModelForTokenClassification
import torch

/Users/lakshminarayananallamothu/Desktop/MAN/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [9]:
tokenizer = AutoTokenizer.from_pretrained("models/")
model = AutoModelForTokenClassification.from_pretrained("models/")

label_list = ["O", "B-SYMPTOM", "I-SYMPTOM", "B-DURATION", "I-DURATION"]


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 7932.06it/s]


In [10]:
def predict(text):
    tokens = text.lower().split()

    inputs = tokenizer(tokens, is_split_into_words=True, return_tensors="pt", truncation=True)
    outputs = model(**inputs)

    preds = torch.argmax(outputs.logits, dim=2)[0].tolist()

    results = []
    for token, pred in zip(tokens, preds[1:len(tokens)+1]):
        results.append((token, label_list[pred]))

    return results


In [11]:
def to_structured_output(predictions):
    symptoms = []
    duration = []

    current = []

    for word, label in predictions:

        if label == "B-SYMPTOM":
            if current:
                symptoms.append(" ".join(current))
                current = []
            current.append(word)

        elif label == "I-SYMPTOM":
            current.append(word)

        elif label == "B-DURATION":
            duration.append(word)

        else:
            if current:
                symptoms.append(" ".join(current))
                current = []

    if current:
        symptoms.append(" ".join(current))

    return {
        "symptoms": symptoms,
        "duration": " ".join(duration)
    }

In [12]:
text = "I feel tensed and have stomach pain since yesterday"

preds = predict(text)

print("Token Predictions:\n")
for word, label in preds:
    print(word, "→", label)

Token Predictions:

i → O
feel → O
tensed → B-SYMPTOM
and → O
have → O
stomach → B-SYMPTOM
pain → I-SYMPTOM
since → O
yesterday → O


In [14]:
structured = to_structured_output(preds)

print("\nFinal Medical Note:\n")
print(structured)


Final Medical Note:

{'symptoms': ['tensed', 'stomach pain'], 'duration': ''}
